# Pipeline de Traducción (Test A/B de Idioma)

Este notebook ejecuta la contingencia definida en la propuesta arquitectónica (V5): traducir el subconjunto de tickets nativos en inglés al español mediante un modelo local *Open Source* de **Hugging Face**. 

El objetivo es generar un dataset en español (`df_final_silver_es.parquet`) que nos permita entrenar un pipeline idéntico al de inglés (Notebook 04) y ejecutar un **Test A/B** para evaluar la degradación semántica causada por la traducción frente al *baseline* nativo.

### Paso 1: Aislamiento del Caché (Buenas Prácticas MLOps)
Antes de importar las librerías de Deep Learning, alteramos el sistema operativo para obligar a Hugging Face a descargar los modelos (que pesan Gigabytes) exclusivamente dentro de la carpeta aislada de nuestro entorno virtual (`.venv`).

In [2]:
import os

# 1. Obtenemos la ruta absoluta de la carpeta actual (donde está este notebook)
current_dir = os.getcwd()

# 2. Definimos la ruta de caché apuntando al interior del entorno virtual
# Subimos un nivel ("..") para salir de la carpeta /notebook y entramos en /.venv
cache_path = os.path.abspath(os.path.join(current_dir, "..", ".venv", "huggingface_cache"))

# 3. Creamos la carpeta si no existe para que no de error
os.makedirs(cache_path, exist_ok=True)

# 4. ¡LA INYECCIÓN CRÍTICA! 
# Modificamos la variable de entorno HF_HOME ANTES de importar nada más.
os.environ['HF_HOME'] = cache_path

print(f"✅ Éxito. Caché de Hugging Face redirigido a:\n{cache_path}")

✅ Éxito. Caché de Hugging Face redirigido a:
d:\MasterEvolve\Proyecto TFM\SITOR\.venv\huggingface_cache


### Paso 2: La Muralla Lógica y Carga de Datos

Traducir un volumen masivo de datos es computacionalmente costoso. Para proteger el trabajo, implementamos un condicional de seguridad: si el archivo traducido final ya existe en nuestro disco duro, el *notebook* detendrá su ejecución inmediatamente para evitar sobrescribir el archivo o perder horas de procesamiento por un click accidental.

Si el archivo no existe, cargaremos la capa Silver, prepararemos el identificador único (`ticket_id`) y el texto concatenado (`full_text`), y dividiremos el dataset en dos:
1. **La reserva nativa:** Los 750 tickets que ya están en español, que guardaremos a un lado.
2. **El bloque de traducción:** Los 23.117 tickets en inglés, a los cuales les extirparemos todas las columnas innecesarias para ahorrar memoria RAM y pasárselos "limpios" al modelo neuronal.

In [3]:
import pandas as pd
import os

ruta_final = "../data/processed/df_final_silver_es.parquet"

# 1. LA MURALLA LÓGICA (Safety Check)
if os.path.exists(ruta_final):
    # Si el archivo existe, lanzamos un error intencionado para detener el Jupyter Notebook
    raise RuntimeError(f"¡ALTO! El archivo {ruta_final} ya existe. No ejecutes este notebook de nuevo a menos que quieras re-traducir todo.")

print("Archivo traducido no detectado. Iniciando el proceso de carga...")

# 2. Carga de la capa Silver completa
df_silver = pd.read_parquet("../data/processed/df_final_silver.parquet")

# 3. Creación del ID único y concatenación de texto (Igual que en Notebook 02)
df_silver.reset_index(drop=True, inplace=True)
df_silver['ticket_id'] = ['TKT-' + str(i).zfill(5) for i in range(1, len(df_silver) + 1)]

# OJO: Aquí concatenamos, pero NO pasamos a minúsculas todavía. 
# Los modelos de traducción funcionan mucho mejor si les respetas las mayúsculas originales.
df_silver['full_text'] = df_silver['subject'] + " " + df_silver['body']
df_silver['full_text'] = df_silver['full_text'].str.strip()

# 4. Separación de idiomas
# Guardamos los españoles en un cajón aparte
df_nativo_es = df_silver[df_silver['language'] == 'es'].copy()

# Cogemos los ingleses para procesarlos
df_nativo_en = df_silver[df_silver['language'] == 'en'].copy()

# 5. Aislamiento de memoria (El bloque de traducción)
# Al traductor no le importan las fechas ni las categorías, solo el ID y el texto.
df_to_translate = df_nativo_en[['ticket_id', 'full_text']].copy()

print(f"✅ Tickets nativos en español reservados: {len(df_nativo_es)}")
print(f"⏳ Tickets en inglés preparados para traducir: {len(df_to_translate)}")

Archivo traducido no detectado. Iniciando el proceso de carga...
✅ Tickets nativos en español reservados: 750
⏳ Tickets en inglés preparados para traducir: 23117


### Paso 3: Carga del Modelo Neuronal y Detección de Hardware

Para realizar la traducción, utilizamos **`Helsinki-NLP/opus-mt-en-es`**. Es un modelo de la familia MarianMT entrenado específicamente para traducir del inglés al español. Es el estándar de oro en la industria del código abierto para tareas de traducción por lotes debido a su excelente equilibrio entre calidad gramatical y peso computacional.

> **Importante (Optimización de Hardware):** El código detectará automáticamente si el ordenador dispone de una tarjeta gráfica dedicada (GPU mediante CUDA) para acelerar el proceso de forma exponencial, o si, en su defecto, debe utilizar el procesador central (CPU).

In [4]:
import torch
from transformers import MarianMTModel, MarianTokenizer

# 1. Detección de Hardware (Aceleración por GPU)
# PyTorch comprueba si el sistema tiene soporte para CUDA (Tarjetas gráficas NVIDIA)
device = "cuda" if torch.cuda.is_available() else "cpu"

print("-" * 50)
print(f"Hardware detectado para la traducción: {device.upper()}")
if device == "cpu":
    print("⚠️ Aviso: Usando el procesador (CPU). La traducción será pesada y tomará su tiempo.")
else:
    print("🚀 ¡Tarjeta gráfica (GPU) detectada! Volaremos en la traducción.")
print("-" * 50)

# 2. Definición del modelo oficial de Hugging Face
model_name = "Helsinki-NLP/opus-mt-en-es"

print(f"\nConectando con Hugging Face para el modelo: '{model_name}'...")
print("Si es la primera vez, descargará varios archivos (puede tardar unos minutos).")
print("Si ya lo tienes, lo cargará desde tu carpeta .venv al instante.")

# 3. Instanciación del Tokenizador y el Modelo
# El tokenizador es el "diccionario" que trocea las frases en inglés en piezas matemáticas
tokenizer = MarianTokenizer.from_pretrained(model_name)

# El modelo es la red neuronal. Usamos .to(device) para enviarlo a la CPU o a la GPU
model = MarianMTModel.from_pretrained(model_name).to(device)

print("\n✅ ¡Tokenizador y Modelo cargados con éxito en memoria!")

d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--------------------------------------------------
Hardware detectado para la traducción: CPU
⚠️ Aviso: Usando el procesador (CPU). La traducción será pesada y tomará su tiempo.
--------------------------------------------------

Conectando con Hugging Face para el modelo: 'Helsinki-NLP/opus-mt-en-es'...
Si es la primera vez, descargará varios archivos (puede tardar unos minutos).
Si ya lo tienes, lo cargará desde tu carpeta .venv al instante.


Loading weights: 100%|██████████| 258/258 [00:00<00:00, 19279.69it/s]



✅ ¡Tokenizador y Modelo cargados con éxito en memoria!


### Paso 4: El Motor de Traducción (Batches y Checkpointing)

Debido al volumen de datos (23.117 tickets) y a que el procesamiento se realiza por CPU, el proceso tomará un tiempo considerable. Para garantizar la estabilidad del sistema, hemos diseñado una arquitectura robusta:

1.  **Procesamiento por Lotes (Batching):** Traduciremos los tickets de 32 en 32. Esto optimiza el uso de la memoria RAM y mantiene la CPU al 100% de eficiencia sin colapsar.
2.  **Ahorro de Memoria (`torch.no_grad`):** Apagamos los gradientes matemáticos del modelo. Le decimos explícitamente a PyTorch que no estamos entrenando, solo traduciendo. Esto reduce el consumo de RAM drásticamente.
3.  **Recuperación ante Desastres (Checkpointing):** Cada 1.000 tickets, el sistema guardará silenciosamente un archivo temporal en disco (`traduccion_temp_checkpoint.csv`). Si el ordenador se reinicia, se cuelga o se va la luz, al volver a ejecutar esta celda **retomará la traducción exactamente donde se quedó**, sin empezar de cero.

In [5]:
from tqdm.auto import tqdm
import time

# Configuración del motor
BATCH_SIZE = 32 # Lote de tickets que se traducen a la vez
CHECKPOINT_INTERVAL = 1000 # Guardar copia de seguridad cada 1000 tickets
checkpoint_path = "../data/processed/traduccion_temp_checkpoint.csv"

# Extraemos los datos a listas planas (mucho más rápidas de iterar que un DataFrame)
textos_en_ingles = df_to_translate['full_text'].tolist()
ids_en_ingles = df_to_translate['ticket_id'].tolist()

translated_data = []
start_idx = 0

# 1. SISTEMA DE RECUPERACIÓN (RESUME)
if os.path.exists(checkpoint_path):
    print("🔍 Archivo de seguridad detectado. Analizando progreso...")
    df_temp = pd.read_csv(checkpoint_path)
    start_idx = len(df_temp)
    # Cargamos lo que ya estaba traducido para no perderlo
    translated_data = df_temp.to_dict('records')
    print(f"🔄 Reanudando la traducción desde el ticket {start_idx} de {len(textos_en_ingles)}")
else:
    print("🚀 Iniciando traducción desde el ticket 0.")

# 2. EL BUCLE PRINCIPAL DE TRADUCCIÓN
# tqdm genera la barra de progreso visual
for i in tqdm(range(start_idx, len(textos_en_ingles), BATCH_SIZE), desc="Traduciendo Tickets"):
    
    # Extraemos el lote actual (ej. del 0 al 32)
    batch_texts = textos_en_ingles[i : i + BATCH_SIZE]
    batch_ids = ids_en_ingles[i : i + BATCH_SIZE]
    
    # PASO A: Tokenización. padding=True asegura que todos tengan la misma longitud rellenando con ceros
    # truncation=True recorta tickets extremadamente largos (max 512 palabras) para que no explote la memoria
    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    
    # PASO B: Inferencia (Traducción matemática pura). no_grad() ahorra el 50% de la RAM.
    with torch.no_grad():
        translated_tokens = model.generate(**inputs)
    
    # PASO C: Descodificación. Pasamos los números resultantes a palabras en español reales.
    translated_batch = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)
    
    # Empaquetamos los resultados emparejados con su ticket_id original
    for t_id, t_text in zip(batch_ids, translated_batch):
        translated_data.append({"ticket_id": t_id, "full_text_es": t_text})
        
    # PASO D: Guardado Automático de Seguridad (El Checkpoint)
    # Comprobamos si hemos cruzado el umbral de los 1000 tickets
    if len(translated_data) % CHECKPOINT_INTERVAL < BATCH_SIZE and len(translated_data) > start_idx:
        pd.DataFrame(translated_data).to_csv(checkpoint_path, index=False)

# 3. GUARDADO FINAL
# Cuando termina el bucle completo, hacemos un último guardado por seguridad
df_translated_final = pd.DataFrame(translated_data)
df_translated_final.to_csv(checkpoint_path, index=False)

print("\n✅ TRADUCCIÓN MASIVA COMPLETADA CON ÉXITO.")

🔍 Archivo de seguridad detectado. Analizando progreso...
🔄 Reanudando la traducción desde el ticket 23117 de 23117


Traduciendo Tickets: 0it [00:00, ?it/s]


✅ TRADUCCIÓN MASIVA COMPLETADA CON ÉXITO.


### Paso 5: Ensamblaje, Auditoría y Exportación Final

En este paso final fusionamos los textos traducidos con sus variables predictivas originales (`queue`, `priority`, etc.). 

Antes de concatenar este bloque con los **750 tickets nativos** en español, aplicamos un `assert` de seguridad (Auditoría de Datos) para garantizar que los nombres de las colas coinciden matemáticamente entre ambos conjuntos. Si no coincidieran, el algoritmo clasificador fallaría en el Notebook 04.

Finalmente, exportamos el resultado a un archivo `.parquet` optimizado y limpiamos los archivos temporales.

In [6]:
import os

print("Iniciando ensamblaje del Dataset en Español...")

# 1. Cargamos el archivo con los textos en español
df_traducido = pd.read_csv("../data/processed/traduccion_temp_checkpoint.csv")

# 2. Hacemos un JOIN con los datos originales usando la clave primaria (ticket_id)
df_es_fusion = pd.merge(df_nativo_en, df_traducido, on='ticket_id', how='inner')

# 3. Reemplazamos el texto en inglés por el traducido y borramos la columna temporal
df_es_fusion['full_text'] = df_es_fusion['full_text_es']
df_es_fusion = df_es_fusion.drop(columns=['full_text_es'])

# 4. AUDITORÍA DE SEGURIDAD MLOps (El check que propusiste)
# Extraemos los valores únicos de la columna 'queue'
etiquetas_ingles = set(df_es_fusion['queue'].unique())
etiquetas_espanol = set(df_nativo_es['queue'].unique())

print(f"Etiquetas (Dataset Traducido): {etiquetas_ingles}")
print(f"Etiquetas (Dataset Nativo): {etiquetas_espanol}")

# Comprobamos matemáticamente que sean idénticas
assert etiquetas_ingles == etiquetas_espanol, "¡ERROR FATAL! Las etiquetas de clasificación no coinciden."
print("✅ Auditoría superada: Las etiquetas coinciden perfectamente.")

# 5. Preparación de los 750 tickets Nativos (Español real)
# Generamos su propia columna 'full_text' para que encaje con la estructura
df_nativo_es['full_text'] = df_nativo_es['subject'] + " " + df_nativo_es['body']
df_nativo_es['full_text'] = df_nativo_es['full_text'].str.strip()

# 6. CONCATENACIÓN MASIVA (La fusión final)
df_final_es = pd.concat([df_es_fusion, df_nativo_es], ignore_index=True)

# Opcional: Borramos subject y body para ahorrar espacio, ya que tenemos el full_text
df_final_es = df_final_es.drop(columns=['subject', 'body'])

# 7. GUARDADO DEFINITIVO
ruta_final = "../data/processed/df_final_silver_es.parquet"
df_final_es.to_parquet(ruta_final, index=False)
print(f"\\n🚀 ¡ÉXITO! Dataset de {df_final_es.shape[0]} tickets guardado en: {ruta_final}")

# 8. Limpieza del ordenador
if os.path.exists("../data/processed/traduccion_temp_checkpoint.csv"):
    os.remove("../data/processed/traduccion_temp_checkpoint.csv")
    print("🧹 Archivo temporal de seguridad eliminado para liberar disco duro.")

Iniciando ensamblaje del Dataset en Español...
Etiquetas (Dataset Traducido): {'Billing and Payments', 'Technical Support', 'Customer Service', 'Sales and Pre-Sales', 'Product Support', 'IT Support', 'Service Outages and Maintenance'}
Etiquetas (Dataset Nativo): {'Billing and Payments', 'Technical Support', 'Customer Service', 'Sales and Pre-Sales', 'Product Support', 'IT Support', 'Service Outages and Maintenance'}
✅ Auditoría superada: Las etiquetas coinciden perfectamente.
\n🚀 ¡ÉXITO! Dataset de 23867 tickets guardado en: ../data/processed/df_final_silver_es.parquet
🧹 Archivo temporal de seguridad eliminado para liberar disco duro.
